# Pontes, institutos, produção e linhas de pesquisa

**Parte A:** matriz de coautorias entre institutos, conexões por instituto, pesquisadores-ponte, atividade por ano.
**Parte B:** nº de artigos no período e linhas de pesquisa por pesquisador. Exige chave gratuita (https://openalex.org/settings/api): guarde em *Secrets* do Colab com o nome `OPENALEX_API_KEY`, ou cole na célula de configuração. Se não houver chave, a Parte B é pulada.

Todas as saídas vão para `/content/figuras`.

In [ ]:
%pip install igraph -q

In [ ]:
import os
try:
    from google.colab import drive
    drive.mount("/content/drive")
except Exception:
    pass  # fora do Colab: defina DATA_DIR

# Chave do OpenAlex (Parte B). Prefira Secrets do Colab; ou descomente e cole a chave:
# os.environ["OPENALEX_API_KEY"] = "COLE_AQUI"
try:
    from google.colab import userdata
    os.environ["OPENALEX_API_KEY"] = userdata.get("OPENALEX_API_KEY")
except Exception:
    pass

## Configuração

In [ ]:
import os, json, time, collections
import numpy as np
import pandas as pd
import igraph as ig

DATA_DIR = os.environ.get("DATA_DIR", "/content/drive/MyDrive")
OUT_DIR = os.environ.get("OUT_DIR", "/content/figuras")
OPENALEX_API_KEY = os.environ.get("OPENALEX_API_KEY", "")   # Parte B
DATA_INICIO, DATA_FIM = "2020-01-01", "2026-08-19"          # mesmo período do artigo
os.makedirs(OUT_DIR, exist_ok=True)

INSTITUTOS = ["IEM", "IEPG", "IESTI", "IFQ", "IMC", "IRN", "ISEE", "IEI"]

nodes = pd.read_csv(os.path.join(DATA_DIR, "nodes.csv"), encoding="utf-8-sig")
edges = pd.read_csv(os.path.join(DATA_DIR, "edges.csv"), encoding="utf-8-sig")


def lista_institutos(valor):
    """'IEM; IMC' -> ['IEM','IMC'];  vazio/'Não identificado' -> []"""
    if pd.isna(valor) or valor.strip() in ("", "Não identificado"):
        return []
    return [x.strip() for x in valor.split(";")]


nodes["il"] = nodes["instituto"].apply(lista_institutos)
idx = {i: k for k, i in enumerate(nodes["id"])}

## Grafo e métricas por nó

In [ ]:
g = ig.Graph(n=len(nodes),
             edges=[(idx[a], idx[b]) for a, b in zip(edges["source"], edges["target"])])
g.es["weight"] = edges["weight"].tolist()
vizinhos = [g.neighbors(i) for i in range(len(nodes))]

nodes["grau"] = g.degree()
nodes["forca"] = g.strength(weights="weight")      # soma dos pesos = nº de coautorias somadas

# Intermediação e pontos de articulação no maior componente (rede não ponderada:
# interessa quem está nos caminhos mínimos, o peso mede intensidade e não distância)
comp = g.connected_components()
gc_ids = sorted(max(comp, key=len))
sub = g.induced_subgraph(gc_ids)
pos_sub = {v: k for k, v in enumerate(gc_ids)}
btw = sub.betweenness()
art = set(gc_ids[a] for a in sub.articulation_points())
nodes["intermediacao"] = np.nan
for v in gc_ids:
    nodes.loc[v, "intermediacao"] = btw[pos_sub[v]]
nodes["ponto_articulacao"] = [i in art for i in range(len(nodes))]

# Institutos presentes na vizinhança (só vizinhos com instituto identificado)
def contagem_inst_vizinhos(i):
    c = collections.Counter()
    for j in vizinhos[i]:
        for s in nodes.at[j, "il"]:
            c[s] += 1
    return c

viz_inst = [contagem_inst_vizinhos(i) for i in range(len(nodes))]
nodes["n_inst_vizinhos"] = [len(c) for c in viz_inst]
nodes["n_vizinhos_nao_ident"] = [sum(1 for j in vizinhos[i] if not nodes.at[j, "il"])
                                 for i in range(len(nodes))]

## A1. Coautorias entre institutos (matriz + mapa de calor)

In [ ]:
M = pd.DataFrame(0, index=INSTITUTOS, columns=INSTITUTOS)
for a, b in zip(edges["source"], edges["target"]):
    for x in nodes.at[idx[a], "il"]:
        for y in nodes.at[idx[b], "il"]:
            if x != y:
                M.loc[x, y] += 1
                M.loc[y, x] += 1
M.to_csv(os.path.join(OUT_DIR, "05_matriz_inter_institutos.csv"), encoding="utf-8-sig")
print(M)

# Mapa de calor (figura para o artigo)
import matplotlib.pyplot as plt
fig, ax = plt.subplots(figsize=(5.2, 4.4), dpi=200)
im = ax.imshow(M.values, cmap="Blues")
ax.set_xticks(range(len(INSTITUTOS))); ax.set_xticklabels(INSTITUTOS, rotation=45, ha="right")
ax.set_yticks(range(len(INSTITUTOS))); ax.set_yticklabels(INSTITUTOS)
for i in range(len(INSTITUTOS)):
    for j in range(len(INSTITUTOS)):
        if i != j:
            ax.text(j, i, int(M.values[i, j]), ha="center", va="center", fontsize=7,
                    color="white" if M.values[i, j] > M.values.max() * 0.55 else "black")
ax.set_title("Coautorias entre institutos (nº de arestas)", fontsize=9)
fig.colorbar(im, ax=ax, shrink=0.8)
plt.tight_layout(); plt.savefig(os.path.join(OUT_DIR, "05_heatmap_inter_institutos.png"), bbox_inches="tight")
plt.show()

## A2. Conexões de cada instituto

In [ ]:
linhas = []
for s in INSTITUTOS:
    membros = set(nodes.index[nodes["il"].apply(lambda l: s in l)])
    intra = inter = ext = 0
    for a, b in zip(edges["source"], edges["target"]):
        ia, ib = idx[a], idx[b]
        if ia in membros or ib in membros:
            outro = ib if ia in membros else ia
            if ia in membros and ib in membros:
                intra += 1
            elif nodes.at[outro, "il"]:
                inter += 1
            else:
                ext += 1
    sub_m = nodes.loc[sorted(membros)]
    linhas.append({
        "instituto": s, "pesquisadores_identificados": len(membros),
        "arestas_internas": intra, "arestas_outros_institutos": inter,
        "arestas_nao_identificados": ext,
        "pct_internas": round(intra / max(intra + inter + ext, 1), 3),
        "grau_mediano": float(sub_m["grau"].median()), "grau_maximo": int(sub_m["grau"].max()),
        "pontos_articulacao_rede_geral": int(sub_m["ponto_articulacao"].sum()),
        "pesq_com_3+_institutos_vizinhos": int((sub_m["n_inst_vizinhos"] >= 3).sum()),
        "pct_forca_top5": round(sub_m["forca"].nlargest(5).sum() / sub_m["forca"].sum(), 3),
    })
conexoes = pd.DataFrame(linhas)
conexoes.to_csv(os.path.join(OUT_DIR, "06_conexoes_por_instituto.csv"), index=False, encoding="utf-8-sig")
print(conexoes.to_string(index=False))

## A3. Pesquisadores-ponte
Critérios: (i) estrutural (intermediação / ponto de articulação) e (ii) institucional (vizinhos em ≥3 institutos).

In [ ]:
# Dois critérios complementares:
#  (i) estrutural: intermediação alta / ponto de articulação (conecta regiões da rede)
#  (ii) institucional: vizinhos em >=3 institutos distintos (conecta institutos)
cols = ["id", "label", "instituto", "grau", "forca", "intermediacao",
        "ponto_articulacao", "n_inst_vizinhos", "n_vizinhos_nao_ident"]
top_btw = nodes.dropna(subset=["intermediacao"]).nlargest(25, "intermediacao")
top_inst = nodes[nodes["il"].apply(len) > 0].nlargest(25, ["n_inst_vizinhos", "grau"])
pontes = pd.concat([top_btw, top_inst]).drop_duplicates("id")[cols]
pontes = pontes.sort_values("intermediacao", ascending=False)
pontes["intermediacao"] = pontes["intermediacao"].round(0)
pontes.to_csv(os.path.join(OUT_DIR, "07_pontes_pesquisadores.csv"), index=False, encoding="utf-8-sig")
print(pontes.head(15).to_string(index=False))

## A4. Atividade por ano
Conta pesquisadores com ao menos uma publicação no ano, não o nº de publicações.

In [ ]:
# ATENÇÃO: isto conta pesquisadores com ao menos uma publicação no ano, não o nº de publicações.
arq_pesq = [f for f in os.listdir(DATA_DIR) if f.startswith("pesquisadores_unifei_todos_institutos")]
if arq_pesq:
    pq = pd.read_csv(os.path.join(DATA_DIR, arq_pesq[0]), encoding="utf-8-sig")
    por_ano = collections.defaultdict(collections.Counter)
    for _, r in pq.iterrows():
        for s in lista_institutos(r["instituto"]):
            for y in str(r["anos"]).split(","):
                if y.strip().isdigit():
                    por_ano[y.strip()][s] += 1
    atividade = pd.DataFrame(por_ano).T.fillna(0).astype(int).sort_index()[INSTITUTOS]
    atividade.index.name = "ano"
    atividade.to_csv(os.path.join(OUT_DIR, "08_atividade_por_ano.csv"), encoding="utf-8-sig")
    print(atividade)
else:
    print("CSV 'pesquisadores_unifei_todos_institutos' não encontrado em DATA_DIR; pulando A4.")

# =====================================================================
# PARTE B - OpenAlex: produção por pesquisador e linhas de pesquisa
# =====================================================================

## Parte B — OpenAlex
Cache em disco: se o limite diário acabar, rode de novo no dia seguinte.

In [ ]:
import requests

BASE = "https://api.openalex.org"
CACHE_PATH = os.path.join(OUT_DIR, "cache_openalex.json")
cache = json.load(open(CACHE_PATH, encoding="utf-8")) if os.path.exists(CACHE_PATH) else {"works": {}, "authors": {}}


def salvar_cache():
    json.dump(cache, open(CACHE_PATH, "w", encoding="utf-8"), ensure_ascii=False)


def _get(url, params, tentativas=5):
    params = dict(params)
    params["api_key"] = OPENALEX_API_KEY
    for t in range(tentativas):
        r = requests.get(url, params=params, timeout=60)
        if r.status_code == 200:
            return r.json()
        if r.status_code in (429, 500, 502, 503):
            time.sleep(2 ** t)          # espera exponencial
            continue
        raise RuntimeError(f"OpenAlex {r.status_code}: {r.text[:300]}")
    raise RuntimeError("OpenAlex: tentativas esgotadas (verifique o limite diário em /rate-limit)")


def contar_trabalhos(author_id):
    """Trabalhos do autor no período, agrupados por tipo (1 chamada por autor)."""
    filtro = (f"authorships.author.id:{author_id},"
              f"from_publication_date:{DATA_INICIO},to_publication_date:{DATA_FIM}")
    resp = _get(f"{BASE}/works", {"filter": filtro, "group_by": "type"})
    por_tipo = {b["key"]: b["count"] for b in resp.get("group_by", [])}
    return {"total": sum(por_tipo.values()), "artigos": por_tipo.get("article", 0), "por_tipo": por_tipo}


def baixar_autores(ids, tamanho=50):
    """Tópicos de vários autores por chamada (filtro openalex:A1|A2|...)."""
    for i in range(0, len(ids), tamanho):
        lote = ids[i:i + tamanho]
        resp = _get(f"{BASE}/authors", {"filter": "openalex:" + "|".join(lote),
                                        "per-page": tamanho, "select": "id,display_name,topics"})
        for a in resp.get("results", []):
            cache["authors"][a["id"].split("/")[-1]] = a.get("topics", [])
        for a in lote:                                  # marca como consultado mesmo sem tópicos
            cache["authors"].setdefault(a, [])
        salvar_cache()


def resumir_topicos(topicos, n=3):
    """Top-n tópicos (por nº de trabalhos) e área predominante."""
    t = sorted(topicos, key=lambda x: x.get("count", 0), reverse=True)[:n]
    nomes = " | ".join(x.get("display_name", "") for x in t)
    campos = collections.Counter()
    for x in topicos:
        campos[(x.get("field") or {}).get("display_name", "?")] += x.get("count", 0)
    campo_top = campos.most_common(1)[0][0] if campos else ""
    dominio = collections.Counter()
    for x in topicos:
        dominio[(x.get("domain") or {}).get("display_name", "?")] += x.get("count", 0)
    dominio_top = dominio.most_common(1)[0][0] if dominio else ""
    return nomes, campo_top, dominio_top

### B1. Coleta (~1.200 chamadas)

In [ ]:
if OPENALEX_API_KEY:
    ident = nodes[nodes["il"].apply(len) > 0]["id"].tolist()
    # pontes já identificadas (mesmo sem instituto) também entram
    ident = list(dict.fromkeys(ident + pontes["id"].tolist()))
    pendentes = [a for a in ident if a not in cache["works"]]
    print(f"{len(ident)} autores; {len(pendentes)} pendentes de contagem de trabalhos")
    for k, a in enumerate(pendentes, 1):
        cache["works"][a] = contar_trabalhos(a)
        if k % 50 == 0:
            salvar_cache(); print(k, "/", len(pendentes))
    salvar_cache()
    baixar_autores([a for a in ident if a not in cache["authors"]])
else:
    print("OPENALEX_API_KEY não definida: Parte B pulada.")

### B2. Tabelas finais

In [ ]:
if OPENALEX_API_KEY:
    linhas = []
    for a, w in cache["works"].items():
        nomes, campo, dominio = resumir_topicos(cache["authors"].get(a, []))
        linhas.append({"id": a, "trabalhos_periodo": w["total"], "artigos_periodo": w["artigos"],
                       "topicos_principais": nomes, "area_predominante": campo, "dominio": dominio})
    prod = nodes.merge(pd.DataFrame(linhas), on="id", how="inner")
    prod[["id", "label", "instituto", "grau", "forca", "trabalhos_periodo", "artigos_periodo",
          "topicos_principais", "area_predominante", "dominio"]].to_csv(
        os.path.join(OUT_DIR, "09_producao_topicos_pesquisadores.csv"), index=False, encoding="utf-8-sig")

    resumo = []
    for s in INSTITUTOS:
        m = prod[prod["il"].apply(lambda l: s in l)]
        areas = m["area_predominante"].replace("", np.nan).dropna().value_counts()
        resumo.append({
            "instituto": s, "n": len(m),
            "artigos_mediana": float(m["artigos_periodo"].median()),
            "artigos_media": round(m["artigos_periodo"].mean(), 1),
            "artigos_max": int(m["artigos_periodo"].max()),
            "pct_artigos_top5": round(m["artigos_periodo"].nlargest(5).sum() / max(m["artigos_periodo"].sum(), 1), 3),
            "corr_grau_x_artigos": round(m["grau"].corr(m["artigos_periodo"], method="spearman"), 2),
            "areas_predominantes": "; ".join(f"{k} ({v})" for k, v in areas.head(3).items()),
        })
    pd.DataFrame(resumo).to_csv(os.path.join(OUT_DIR, "10_resumo_por_instituto.csv"), index=False, encoding="utf-8-sig")
    print(pd.DataFrame(resumo).to_string(index=False))

    pontes_prod = pontes.merge(
        prod[["id", "trabalhos_periodo", "artigos_periodo", "topicos_principais", "area_predominante"]],
        on="id", how="left")
    pontes_prod.to_csv(os.path.join(OUT_DIR, "11_pontes_com_producao.csv"), index=False, encoding="utf-8-sig")
    print(pontes_prod.head(15).to_string(index=False))